# Netflix Movies Data Analysis using Exploratory Data Analysis (EDA)

## Project Goal

This notebook analyzes Netflix movie data to determine whether movie durations are decreasing over time and to identify patterns across genres, release years, countries, and content types.

The workflow is written for beginners, but it follows a professional data analyst structure: load data, inspect it, clean it, transform it, visualize it, and summarize business insights.

## 1. Import Libraries

**Objective:** Import the Python libraries needed for data analysis and visualization.

**Approach:** Use Pandas and NumPy for data handling, and Matplotlib and Seaborn for charts.

**Findings:** This section prepares the notebook environment. No dataset findings are produced yet.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set a clean chart style for the full notebook.
sns.set_theme(style="whitegrid", palette="deep")

# Make charts large enough for portfolio screenshots.
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 15
plt.rcParams["axes.labelsize"] = 12

## 2. Load the Dataset

**Objective:** Load the Netflix dataset from a CSV file named `netflix_data.csv`.

**Approach:** The notebook first looks for the file in `data/raw/netflix_data.csv`. If it is not found there, it checks the project root.

**Findings:** After loading the file, we inspect the number of rows and columns to understand the dataset size.

In [ ]:
project_root = Path.cwd()
data_path = project_root / "data" / "raw" / "netflix_data.csv"
fallback_path = project_root / "netflix_data.csv"

if data_path.exists():
    csv_path = data_path
elif fallback_path.exists():
    csv_path = fallback_path
else:
    raise FileNotFoundError(
        "Place netflix_data.csv in data/raw/ or in the project root."
    )

df = pd.read_csv(csv_path)

print(f"Dataset loaded from: {csv_path}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

df.head()

## 3. Initial Data Inspection

**Objective:** Understand the dataset structure before cleaning.

**Approach:** Review column names, data types, missing values, duplicate rows, and the distribution of content types.

**Findings:** This step reveals data quality issues that need to be handled before analysis.

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False))

print(f"\nDuplicate rows: {df.duplicated().sum():,}")

if "type" in df.columns:
    print("\nContent type distribution:")
    display(df["type"].value_counts())

## 4. Data Cleaning

**Objective:** Prepare a clean movie-only dataset for EDA.

**Approach:** Standardize column names, support `listed_in` as a genre column if needed, select important columns, filter only movies, fill missing categorical values, convert release year to numeric, and convert duration into integer minutes.

**Findings:** Clean data makes the analysis reliable. Movie duration must be numeric before we can calculate averages, correlations, or trend lines.

In [ ]:
important_columns = [
    "title",
    "type",
    "genre",
    "release_year",
    "duration",
    "country",
]

# Standardize column names for easier coding.
clean_df = df.copy()
clean_df.columns = (
    clean_df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# Some Netflix datasets use listed_in instead of genre.
if "genre" not in clean_df.columns and "listed_in" in clean_df.columns:
    clean_df = clean_df.rename(columns={"listed_in": "genre"})

missing_columns = [
    column for column in important_columns if column not in clean_df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Select only the columns needed for this project.
clean_df = clean_df[important_columns].copy()

# Keep only Movie records.
movies = clean_df[clean_df["type"].str.lower().eq("movie")].copy()

# Fill missing values in categorical columns.
movies["title"] = movies["title"].fillna("Unknown Title")
movies["genre"] = movies["genre"].fillna("Unknown")
movies["country"] = movies["country"].fillna("Unknown")

# Convert release year and duration into numeric columns.
movies["release_year"] = pd.to_numeric(
    movies["release_year"], errors="coerce"
)
movies["duration_minutes"] = (
    movies["duration"].astype(str).str.extract(r"(\d+)")[0]
)
movies["duration_minutes"] = pd.to_numeric(
    movies["duration_minutes"], errors="coerce"
)

# Remove rows where the key analysis fields are missing.
movies = movies.dropna(subset=["release_year", "duration_minutes"])

movies["release_year"] = movies["release_year"].astype(int)
movies["duration_minutes"] = movies["duration_minutes"].astype(int)

movies.head()

## 5. Feature Engineering

**Objective:** Create additional columns that make analysis easier.

**Approach:** Extract the primary genre from genre lists and create a decade column from release year.

**Findings:** These features allow us to compare movie durations by genre and decade.

In [ ]:
# Use the first listed genre as the primary genre for grouped analysis.
movies["primary_genre"] = (
    movies["genre"].astype(str).str.split(",").str[0].str.strip()
)

# Create decade values such as 1990, 2000, and 2010.
movies["decade"] = (movies["release_year"] // 10) * 10

print(f"Clean movie records: {len(movies):,}")
print(f"Release year range: {movies['release_year'].min()}-"
      f"{movies['release_year'].max()}")
print(f"Average duration: {movies['duration_minutes'].mean():.1f} minutes")

movies[[
    "title",
    "release_year",
    "duration_minutes",
    "primary_genre",
    "decade",
]].head()

## 6. Scatter Plot: Movie Duration vs Release Year

**Objective:** Check whether movie durations are increasing, decreasing, or staying stable over time.

**Approach:** Plot each movie as one point and add a red trend line using regression.

**Findings:** If the trend line slopes downward, newer movies tend to be shorter. If it slopes upward, newer movies tend to be longer.

In [ ]:
plt.figure(figsize=(12, 6))
sns.regplot(
    data=movies,
    x="release_year",
    y="duration_minutes",
    scatter_kws={"alpha": 0.35},
    line_kws={"color": "red", "linewidth": 2},
)
plt.title("Netflix Movie Duration vs Release Year")
plt.xlabel("Release Year")
plt.ylabel("Duration (Minutes)")
plt.tight_layout()
plt.savefig("reports/figures/duration_vs_release_year.png", dpi=300)
plt.show()

## 7. Histogram: Distribution of Movie Durations

**Objective:** Understand the most common duration range for Netflix movies.

**Approach:** Use a histogram with a density curve to show how durations are distributed.

**Findings:** The tallest bars show the most common movie length range. Long tails may indicate unusually short or long movies.

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(movies["duration_minutes"], bins=30, kde=True, color="#2a9d8f")
plt.title("Distribution of Netflix Movie Durations")
plt.xlabel("Duration (Minutes)")
plt.ylabel("Number of Movies")
plt.tight_layout()
plt.savefig("reports/figures/duration_histogram.png", dpi=300)
plt.show()

## 8. Genre Distribution

**Objective:** Identify the most common movie genres on Netflix.

**Approach:** Count the top 10 primary genres and visualize them with a horizontal bar chart.

**Findings:** The longest bars represent genres that appear most frequently in the movie catalog.

In [ ]:
top_genres = movies["primary_genre"].value_counts().head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_genres.values, y=top_genres.index, palette="viridis")
plt.title("Top 10 Netflix Movie Genres")
plt.xlabel("Number of Movies")
plt.ylabel("Primary Genre")
plt.tight_layout()
plt.savefig("reports/figures/genre_distribution.png", dpi=300)
plt.show()

## 9. Movies Released Each Year

**Objective:** Analyze how Netflix movie release volume changes over time.

**Approach:** Count movie records by release year and plot the yearly trend.

**Findings:** Peaks show years with many movies in the dataset, while dips show lower representation.

In [ ]:
yearly_counts = movies["release_year"].value_counts().sort_index()

plt.figure(figsize=(14, 6))
sns.lineplot(x=yearly_counts.index, y=yearly_counts.values, marker="o")
plt.title("Netflix Movies Released Each Year")
plt.xlabel("Release Year")
plt.ylabel("Number of Movies")
plt.tight_layout()
plt.savefig("reports/figures/movies_per_year.png", dpi=300)
plt.show()

## 10. Boxplot: Duration by Genre

**Objective:** Compare duration ranges across common genres.

**Approach:** Use a boxplot for the top 10 genres to show median duration, spread, and outliers.

**Findings:** Genres with lower medians tend to have shorter movies. Wide boxes suggest more variation within a genre.

In [ ]:
top_10_genres = movies["primary_genre"].value_counts().head(10).index
genre_subset = movies[movies["primary_genre"].isin(top_10_genres)]

plt.figure(figsize=(14, 7))
sns.boxplot(
    data=genre_subset,
    x="duration_minutes",
    y="primary_genre",
    palette="Set2",
)
plt.title("Movie Duration by Top Genres")
plt.xlabel("Duration (Minutes)")
plt.ylabel("Primary Genre")
plt.tight_layout()
plt.savefig("reports/figures/duration_by_genre_boxplot.png", dpi=300)
plt.show()

## 11. Correlation Heatmap

**Objective:** Measure relationships between numeric variables.

**Approach:** Calculate correlations between release year, duration, and decade, then display them in a heatmap.

**Findings:** A negative correlation between release year and duration would support the idea that movies are getting shorter.

In [ ]:
numeric_columns = movies[["release_year", "duration_minutes", "decade"]]

plt.figure(figsize=(8, 5))
sns.heatmap(numeric_columns.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig("reports/figures/correlation_heatmap.png", dpi=300)
plt.show()

## 12. Advanced Analysis: Genre-Wise Duration Trends

**Objective:** Understand whether duration trends differ by genre.

**Approach:** Select the five most common primary genres, calculate average duration by release year, and plot each genre as a separate trend line.

**Findings:** This view shows whether some genres are becoming shorter or longer faster than others.

In [ ]:
top_5_genres = movies["primary_genre"].value_counts().head(5).index
genre_trend_data = movies[movies["primary_genre"].isin(top_5_genres)]
genre_trend_data = (
    genre_trend_data.groupby(["release_year", "primary_genre"])[
        "duration_minutes"
    ]
    .mean()
    .reset_index()
)

plt.figure(figsize=(14, 7))
sns.lineplot(
    data=genre_trend_data,
    x="release_year",
    y="duration_minutes",
    hue="primary_genre",
    marker="o",
)
plt.title("Genre-Wise Average Movie Duration Trend")
plt.xlabel("Release Year")
plt.ylabel("Average Duration (Minutes)")
plt.legend(title="Primary Genre")
plt.tight_layout()
plt.savefig("reports/figures/genre_duration_trends.png", dpi=300)
plt.show()

## 13. Average Duration by Decade

**Objective:** Find which decade produced the longest Netflix movies on average.

**Approach:** Group movies by decade and calculate the mean duration for each decade.

**Findings:** The tallest bar represents the decade with the longest average movie duration.

In [ ]:
decade_duration = (
    movies.groupby("decade")["duration_minutes"].mean().reset_index()
)

plt.figure(figsize=(12, 6))
sns.barplot(
    data=decade_duration,
    x="decade",
    y="duration_minutes",
    palette="magma",
)
plt.title("Average Netflix Movie Duration by Decade")
plt.xlabel("Decade")
plt.ylabel("Average Duration (Minutes)")
plt.tight_layout()
plt.savefig("reports/figures/average_duration_by_decade.png", dpi=300)
plt.show()

decade_duration.sort_values("duration_minutes", ascending=False)

## 14. Top 10 Longest Movies

**Objective:** Identify the longest movies in the dataset.

**Approach:** Sort movies by duration in descending order and display the top 10 records.

**Findings:** These records often represent special cases and may affect averages if they are extreme.

In [ ]:
top_10_longest = movies.nlargest(10, "duration_minutes")[
    ["title", "release_year", "primary_genre", "country", "duration_minutes"]
]

top_10_longest

## 15. Outlier Detection

**Objective:** Detect unusually short or unusually long movies.

**Approach:** Use the IQR method. Values below Q1 - 1.5 x IQR or above Q3 + 1.5 x IQR are marked as outliers.

**Findings:** Outliers are important because they can influence averages and trend lines.

In [ ]:
q1 = movies["duration_minutes"].quantile(0.25)
q3 = movies["duration_minutes"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = movies[
    (movies["duration_minutes"] < lower_bound)
    | (movies["duration_minutes"] > upper_bound)
].copy()

print(f"Q1: {q1:.1f}")
print(f"Q3: {q3:.1f}")
print(f"IQR: {iqr:.1f}")
print(f"Lower bound: {lower_bound:.1f}")
print(f"Upper bound: {upper_bound:.1f}")
print(f"Outliers found: {len(outliers):,}")

outliers.sort_values("duration_minutes", ascending=False)[
    ["title", "release_year", "primary_genre", "duration_minutes"]
].head(15)

## 16. Final Insight Summary

**Objective:** Convert the analysis into clear business insights.

**Approach:** Calculate summary statistics for duration trends, genre patterns, decade averages, and content volume.

**Findings:** The following cell generates a data-driven summary based on your CSV file.

In [ ]:
duration_year_correlation = movies["release_year"].corr(
    movies["duration_minutes"]
)
average_by_decade = movies.groupby("decade")["duration_minutes"].mean()
average_by_genre = (
    movies.groupby("primary_genre")["duration_minutes"]
    .mean()
    .sort_values()
)
movies_by_decade = movies["decade"].value_counts().sort_index()

longest_decade = average_by_decade.idxmax()
shortest_genres = average_by_genre.head(5)
most_active_decade = movies_by_decade.idxmax()

print("Final Netflix Movie Insights")
print("=" * 30)
print(f"Total movies analyzed: {len(movies):,}")
print(f"Average movie duration: {movies['duration_minutes'].mean():.1f} minutes")
print(
    "Correlation between release year and duration: "
    f"{duration_year_correlation:.3f}"
)

if duration_year_correlation < -0.1:
    print("Movies appear to be getting shorter over time.")
elif duration_year_correlation > 0.1:
    print("Movies appear to be getting longer over time.")
else:
    print("Movie durations appear mostly stable over time.")

print(f"\nDecade with longest average movies: {longest_decade}s")
print(f"Decade with most movies in this dataset: {most_active_decade}s")

print("\nGenres with shortest average durations:")
display(shortest_genres.round(1))

print("\nAverage duration by decade:")
display(average_by_decade.round(1))

## 17. Business Conclusion

**Objective:** Summarize what the EDA means from a business and content strategy perspective.

**Approach:** Use the charts and summary statistics above to connect movie duration trends with genre and release-year patterns.

**Conclusion:**

- If the release-year trend line slopes downward and the correlation is negative, the dataset supports the idea that Netflix movies have become shorter over time.
- Shorter average durations in specific genres may suggest that audiences prefer quicker viewing experiences for those categories.
- The decade-level chart helps identify periods where longer films were more common.
- The yearly movie count trend shows how Netflix catalog volume changes across release years.
- Outlier detection helps separate unusual movies from normal catalog behavior, making the average-duration analysis more trustworthy.

Overall, this project demonstrates how EDA can turn a simple CSV file into practical insights about content trends, audience experience, and catalog strategy.